# 01 — Data overview

This notebook loads the packaged synthetic generator maintenance dataset and gives you a quick tour of the tables.

## What is inside
- `asset_master.csv`: one row per generator
- `pm_events.csv`: preventive maintenance events
- `failure_events.csv`: corrective or unplanned failures
- `pm_failure_linked.csv`: PM events linked to the next failure for that asset
- `telemetry_weekly.csv`: weekly telemetry snapshots with labels
- `business_impact.csv`: downtime and cost impact estimates

## Why this package exists
This package is designed for a **quick demo** of PM-to-failure analysis and predictive maintenance. The included data is synthetic but structured to look like a real generator maintenance program.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path('..')
DATA = BASE / 'data' / 'processed'
OUT = BASE / 'outputs'


In [ ]:
assets = pd.read_csv(DATA / 'asset_master.csv')
pm = pd.read_csv(DATA / 'pm_events.csv', parse_dates=['scheduled_date', 'completed_date', 'pm_date'])
fail = pd.read_csv(DATA / 'failure_events.csv', parse_dates=['failure_date', 'ticket_open_date', 'ticket_close_date'])
linked = pd.read_csv(DATA / 'pm_failure_linked.csv', parse_dates=['pm_date', 'next_failure_date'])
telemetry = pd.read_csv(DATA / 'telemetry_weekly.csv', parse_dates=['timestamp'])
impact = pd.read_csv(DATA / 'business_impact.csv', parse_dates=['event_date'])

for name, df in [('assets', assets), ('pm', pm), ('fail', fail), ('linked', linked), ('telemetry', telemetry), ('impact', impact)]:
    print(f'{name:10s} -> {df.shape}')

In [ ]:
assets.head()

In [ ]:
pm.head()

In [ ]:
telemetry.head()

## Quick data quality checks

These checks help confirm that keys, dates, and labels look healthy before you move into modeling or dashboards.

In [ ]:
checks = {
    'duplicate asset ids': int(assets['asset_id'].duplicated().sum()),
    'pm rows missing asset_id': int(pm['asset_id'].isna().sum()),
    'failure rows missing asset_id': int(fail['asset_id'].isna().sum()),
    'telemetry rows missing label': int(telemetry['failure_within_30d'].isna().sum()),
}
checks

In [ ]:
summary = pd.DataFrame({
    'table': ['assets', 'pm', 'failures', 'linked_pm_failures', 'telemetry', 'business_impact'],
    'rows': [len(assets), len(pm), len(fail), len(linked), len(telemetry), len(impact)]
})
summary